# Comparing Layout Clustering vs Standalone Dripper

**Machine**: dgx-a100-02 (10.184.206.11)  
**Dataset**: CC-MAIN-2025-26 smoke test  

| | Run A | Run B |
|---|---|---|
| **Mode** | Dripper + Layout Clustering | Standalone Dripper |
| **Job ID** | 334943 | 334945 |
| **LLM calls** | 1 per cluster representative (rest templated) | 1 per page |

**Sections**

0. Setup  
1. Load data  
2. LLM call efficiency  
3. Throughput & cost  
4. Quality: F1 comparison  
5. Per-host analysis  
6. Cluster size distribution  
7. Example content comparison  
8. Summary scorecard

## 0. Setup

In [ ]:
%matplotlib inline
import sys, os, re, json, time, warnings
from pathlib import Path
from collections import Counter

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# Configurable paths
# ---------------------------------------------------------------------------
CURATOR_REPO = "/raid/vjawa/nemo-curator-adlr-mm/submodules/Curator"

RUN_A_DIR = "/lustre/fsw/portfolios/llmservice/users/vjawa/dripper_cc_main_2025_26_smoke/334943"   # with clustering
RUN_B_DIR = "/lustre/fsw/portfolios/llmservice/users/vjawa/dripper_cc_main_2025_26_smoke/334945"   # standalone Dripper

# Cluster manifest produced by layout precompute job — choose one:
MANIFEST_DIR = "/lustre/fsw/portfolios/llmservice/users/vjawa/nemo_curator_dripper_layout_clustering_20260611_194849/output_00"
# MANIFEST_DIR = "/raid/vjawa/dripper_tutorial"   # DGX copy (faster I/O)

# ---------------------------------------------------------------------------
sys.path.insert(0, CURATOR_REPO)

import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.dpi"] = 110

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.float_format", "{:.4f}".format)


def read_parquet(path):
    """Use ParquetFile directly — avoids ParquetDataset buffer error on pyarrow 23."""
    return pq.ParquetFile(str(path)).read().to_pandas()


def load_json_safe(path):
    """Load JSON; return {} if not yet written."""
    try:
        with open(path) as f:
            return json.load(f)
    except FileNotFoundError:
        return {}
    except Exception as e:
        print(f"  Warning reading {path}: {e}")
        return {}


def load_parquet_safe(path, label):
    """Load a parquet file; print a clear message if not ready yet."""
    try:
        df = read_parquet(path)
        print(f"  [{label}] {len(df):,} rows  ← {path}")
        return df
    except FileNotFoundError:
        print(f"  [{label}] NOT FOUND — {path}")
        print(f"    (job may still be running; re-run this cell when complete)")
        return None
    except Exception as e:
        print(f"  [{label}] ERROR: {e}")
        return None


def get_metric(m, *keys, default=0):
    """Retrieve a metric by any of several possible key names."""
    for k in keys:
        if k in m:
            return m[k]
    return default


print("Setup OK")
print(f"  Run A : {RUN_A_DIR}")
print(f"  Run B : {RUN_B_DIR}")
print(f"  Manifest : {MANIFEST_DIR}")

## 1. Load Data

In [ ]:
def find_file(run_dir, names):
    """Return the first matching path under run_dir, or None."""
    for name in names:
        # direct
        p = Path(run_dir) / name
        if p.exists():
            return p
        # one level deep (e.g. output/ subdir)
        for child in sorted(Path(run_dir).iterdir()):
            if child.is_dir():
                q = child / name
                if q.exists():
                    return q
    return None


print("Loading Run A (with clustering)...")
ra_results_path = find_file(RUN_A_DIR, ["dripper_results.parquet"])
ra_metrics_path = find_file(RUN_A_DIR, ["metrics.json", "dripper_metrics.json"])
run_a    = load_parquet_safe(ra_results_path, "A results") if ra_results_path else None
metrics_a = load_json_safe(ra_metrics_path) if ra_metrics_path else {}
if not metrics_a:
    print(f"  [A metrics] not found in {RUN_A_DIR}")
else:
    print(f"  [A metrics] keys: {list(metrics_a.keys())}")

print()
print("Loading Run B (standalone Dripper)...")
rb_results_path = find_file(RUN_B_DIR, ["dripper_results.parquet"])
rb_metrics_path = find_file(RUN_B_DIR, ["metrics.json", "dripper_metrics.json"])
run_b    = load_parquet_safe(rb_results_path, "B results") if rb_results_path else None
metrics_b = load_json_safe(rb_metrics_path) if rb_metrics_path else {}
if not metrics_b:
    print(f"  [B metrics] not found in {RUN_B_DIR}")
else:
    print(f"  [B metrics] keys: {list(metrics_b.keys())}")

print()
print("Loading cluster manifest...")
manifest = load_parquet_safe(
    Path(MANIFEST_DIR) / "layout_precompute_manifest.parquet", "manifest"
)
if manifest is not None and "url_host_name" in manifest.columns:
    print(f"  {manifest['url_host_name'].nunique()} unique hosts")

In [ ]:
# Quick schema inspection
for label, df in [("Run A", run_a), ("Run B", run_b), ("Manifest", manifest)]:
    if df is not None:
        print(f"{label} columns ({len(df.columns)}): {list(df.columns)}")
        print()

if run_a is not None and run_b is not None:
    overlap = set(run_a["url"]) & set(run_b["url"])
    print(f"URL overlap A ∩ B: {len(overlap):,}")
    print(f"  A only: {len(set(run_a['url']) - set(run_b['url'])):,}")
    print(f"  B only: {len(set(run_b['url']) - set(run_a['url'])):,}")

## 2. LLM Call Efficiency

Layout clustering avoids one LLM call per clustered page — only the representative is processed by the model; siblings receive the template result without any GPU inference.

Key `metrics.json` fields:
- `llm_request_pages` — pages that triggered an actual LLM call
- `layout_template_saved_call_pages` — pages whose result came from template propagation  
- `total_tokens` — total prompt + completion tokens

In [ ]:
# Pull from metrics, falling back to row counts when jobs are still running
total_pages_a = get_metric(metrics_a, "total_pages", "num_pages",
                            default=len(run_a) if run_a is not None else 0)
total_pages_b = get_metric(metrics_b, "total_pages", "num_pages",
                            default=len(run_b) if run_b is not None else 0)

llm_calls_a   = get_metric(metrics_a, "llm_request_pages", "llm_calls", "num_llm_calls",
                            default=0)
llm_calls_b   = get_metric(metrics_b, "llm_request_pages", "llm_calls", "num_llm_calls",
                            default=total_pages_b)  # standalone = every page

saved_a       = get_metric(metrics_a, "layout_template_saved_call_pages",
                            "templated_pages", "propagated_pages", default=0)
tokens_a      = get_metric(metrics_a, "total_tokens", "total_input_tokens", default=0)
tokens_b      = get_metric(metrics_b, "total_tokens", "total_input_tokens", default=0)

# Derived
call_reduction_pct  = (1 - llm_calls_a / llm_calls_b)  * 100 if llm_calls_b > 0 else 0
token_reduction_pct = (1 - tokens_a    / tokens_b)      * 100 if tokens_b    > 0 else 0
calls_saved         = llm_calls_b - llm_calls_a
tokens_saved        = tokens_b    - tokens_a

# Print summary table
W = 36
print(f"{'Metric':<{W}}  {'Run A (clustering)':>22}  {'Run B (standalone)':>22}")
print("-" * (W + 50))

def fmti(v):
    return f"{v:>22,}" if v else f"{'pending':>22}"

def fmts(v):
    return f"{v:>22}" if v else f"{'pending':>22}"

print(f"{'Total pages':<{W}}{fmti(total_pages_a)}{fmti(total_pages_b)}")
print(f"{'LLM calls (GPU)':<{W}}{fmti(llm_calls_a)}{fmti(llm_calls_b)}")
print(f"{'Templated (no GPU)':<{W}}{fmti(saved_a)}{'N/A':>22}")
print(f"{'Total tokens':<{W}}{fmti(tokens_a)}{fmti(tokens_b)}")
print(f"{'Call reduction vs standalone':<{W}}{f'{call_reduction_pct:.1f}%':>22}{'baseline':>22}")
print(f"{'Token reduction vs standalone':<{W}}{f'{token_reduction_pct:.1f}%':>22}{'baseline':>22}")
print()
print(f"Calls saved: {calls_saved:,}   Tokens saved: {tokens_saved:,}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
runs   = ["Run A\n(clustering)", "Run B\n(standalone)"]
colors = ["#5cb85c", "#d9534f"]

# Panel 1: pages vs LLM calls (grouped)
ax = axes[0]
x, w = np.arange(2), 0.35
b1 = ax.bar(x - w/2, [total_pages_a, total_pages_b], width=w,
            label="Total pages", color="steelblue", alpha=0.85)
b2 = ax.bar(x + w/2, [llm_calls_a,   llm_calls_b],  width=w,
            label="LLM calls",   color="#f0ad4e",   alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(runs)
ax.set_title("Pages vs LLM Calls")
ax.set_ylabel("Count")
ax.legend(fontsize=8)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:,.0f}"))
for b in list(b1) + list(b2):
    h = b.get_height()
    if h > 0:
        ax.text(b.get_x() + b.get_width()/2, h * 1.01, f"{h:,.0f}",
                ha="center", va="bottom", fontsize=7)

# Panel 2: call reduction stacked
ax = axes[1]
if saved_a > 0 and total_pages_a > 0:
    ax.bar(["Run A\n(clustering)"], [llm_calls_a],
           color="#d9534f", label="LLM calls (GPU)")
    ax.bar(["Run A\n(clustering)"], [saved_a],
           bottom=[llm_calls_a], color="#5cb85c", label="Templated (no GPU)")
    ax.bar(["Run B\n(standalone)"], [llm_calls_b], color="#d9534f")
    ax.legend(fontsize=8)
else:
    ax.bar(runs, [llm_calls_a, llm_calls_b], color=colors, edgecolor="black", linewidth=0.5)
    for i, v in enumerate([llm_calls_a, llm_calls_b]):
        if v > 0:
            ax.text(i, v * 1.01, f"{v:,}", ha="center", va="bottom",
                    fontsize=9, fontweight="bold")
ax.set_title(f"LLM Calls ({call_reduction_pct:.1f}% reduction)" if call_reduction_pct else "LLM Calls")
ax.set_ylabel("Pages")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:,.0f}"))

# Panel 3: tokens
ax = axes[2]
ax.bar(runs, [tokens_a, tokens_b], color=colors, edgecolor="black", linewidth=0.5)
ax.set_title(f"Total Tokens ({token_reduction_pct:.1f}% reduction)" if token_reduction_pct else "Total Tokens")
ax.set_ylabel("Tokens")
ax.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda v, _: f"{v/1e6:.1f}M" if v >= 1e6 else f"{v/1e3:.0f}K" if v >= 1e3 else f"{v:.0f}")
)
for i, v in enumerate([tokens_a, tokens_b]):
    if v > 0:
        label = f"{v/1e6:.1f}M" if v >= 1e6 else f"{v/1e3:.0f}K"
        ax.text(i, v * 1.01, label, ha="center", va="bottom",
                fontsize=9, fontweight="bold")

fig.suptitle("LLM Call Efficiency — Clustering vs Standalone", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 3. Throughput & Cost

Measured pages/s → projected H100-hours for the full CC-MAIN-2025-26 snapshot (~2.4 B pages).

In [ ]:
FULL_SNAPSHOT_PAGES = 2_400_000_000

elapsed_a  = get_metric(metrics_a, "elapsed_s", "wall_time_s", "total_elapsed_s", default=0)
elapsed_b  = get_metric(metrics_b, "elapsed_s", "wall_time_s", "total_elapsed_s", default=0)
gpus_a     = get_metric(metrics_a, "num_gpus", "gpus", default=8)
gpus_b     = get_metric(metrics_b, "num_gpus", "gpus", default=8)

tput_a = total_pages_a / elapsed_a if elapsed_a > 0 else 0
tput_b = total_pages_b / elapsed_b if elapsed_b > 0 else 0

# Projected cost: scale measured seconds → full snapshot → GPU-hours
h100h_a = ((FULL_SNAPSHOT_PAGES / tput_a) / 3600 * gpus_a) if tput_a > 0 else 0
h100h_b = ((FULL_SNAPSHOT_PAGES / tput_b) / 3600 * gpus_b) if tput_b > 0 else 0
cost_reduction_pct = (1 - h100h_a / h100h_b) * 100 if h100h_b > 0 else 0

rows = [
    ["Elapsed (s)",                f"{elapsed_a:,.0f}" if elapsed_a else "pending",
                                    f"{elapsed_b:,.0f}" if elapsed_b else "pending"],
    ["Throughput (pages/s)",        f"{tput_a:.2f}"     if tput_a else "pending",
                                    f"{tput_b:.2f}"     if tput_b else "pending"],
    ["GPU count",                   str(gpus_a),  str(gpus_b)],
    ["Projected H100-hours (full)", f"{h100h_a:,.0f}"   if h100h_a else "pending",
                                    f"{h100h_b:,.0f}"   if h100h_b else "pending"],
    ["Cost reduction vs standalone",f"{cost_reduction_pct:.1f}%" if cost_reduction_pct else "pending",
                                    "baseline"],
]
df_perf = pd.DataFrame(rows, columns=["Metric", "Run A (clustering)", "Run B (standalone)"])
df_perf = df_perf.set_index("Metric")
print(df_perf.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
runs   = ["Run A\n(clustering)", "Run B\n(standalone)"]
colors = ["#5cb85c", "#d9534f"]

# Panel 1: throughput
ax = axes[0]
if tput_a > 0 or tput_b > 0:
    bars = ax.bar(runs, [tput_a, tput_b], color=colors, edgecolor="black", linewidth=0.5)
    for bar, v in zip(bars, [tput_a, tput_b]):
        if v > 0:
            ax.text(bar.get_x() + bar.get_width()/2, v * 1.01,
                    f"{v:.2f}", ha="center", va="bottom", fontsize=10, fontweight="bold")
    ax.set_ylabel("pages / second")
    ax.set_title("Throughput")
else:
    ax.text(0.5, 0.5, "Throughput pending\n(jobs may be running)",
            ha="center", va="center", transform=ax.transAxes, fontsize=11, color="gray")
    ax.set_title("Throughput")

# Panel 2: H100-hours
ax = axes[1]
if h100h_a > 0 or h100h_b > 0:
    bars = ax.bar(runs, [h100h_a, h100h_b], color=colors, edgecolor="black", linewidth=0.5)
    for bar, v in zip(bars, [h100h_a, h100h_b]):
        if v > 0:
            ax.text(bar.get_x() + bar.get_width()/2, v * 1.01,
                    f"{v/1000:.0f}K", ha="center", va="bottom", fontsize=10, fontweight="bold")
    ax.set_ylabel("Projected H100-hours")
    ax.set_title(f"H100-hours (full 2.4B page snapshot)"
                 + (f" — {cost_reduction_pct:.1f}% cheaper" if cost_reduction_pct else ""))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v/1000:.0f}K"))
else:
    ax.text(0.5, 0.5, "Cost data pending",
            ha="center", va="center", transform=ax.transAxes, fontsize=11, color="gray")
    ax.set_title("Projected H100-hours")

plt.suptitle("Throughput & Projected Cost", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

if h100h_a > 0 and h100h_b > 0:
    print(f"H100-hours saved: {h100h_b - h100h_a:,.0f}  ({cost_reduction_pct:.1f}%)")

## 4. Quality: F1 Comparison

We merge Run A and Run B on `url`, then compute `_token_f1` between:
- Run A `dripper_content` — extracted via clustering + template propagation  
- Run B `dripper_content` — standalone LLM (treated as ground truth)

Token bag-of-words F1 = harmonic mean of token precision and recall.  
Target: mean F1 ≥ 0.95.

In [ ]:
try:
    from nemo_curator.stages.text.experimental.dripper.stage import _token_f1
    print("_token_f1 loaded from nemo_curator")
except ImportError as e:
    print(f"Import failed ({e}) — using local fallback.")

    def _token_f1(pred: str, ref: str) -> float:
        """Token bag-of-words F1 (fallback)."""
        if not pred and not ref:
            return 1.0
        if not pred or not ref:
            return 0.0
        pred_toks = Counter(re.findall(r"\w+", pred.lower()))
        ref_toks  = Counter(re.findall(r"\w+", ref.lower()))
        common    = sum((pred_toks & ref_toks).values())
        if common == 0:
            return 0.0
        prec = common / sum(pred_toks.values())
        rec  = common / sum(ref_toks.values())
        return 2 * prec * rec / (prec + rec)

In [ ]:
f1_df        = None
is_prop_col  = None

if run_a is None or run_b is None:
    print("Run A or Run B not loaded — skipping F1 analysis.")
    print("Re-run Section 1 once both jobs complete.")
else:
    # Find content columns
    def find_col(df, candidates):
        for c in candidates:
            if c in df.columns:
                return c
        return None

    content_col_a = find_col(run_a, ["dripper_content", "main_content", "content"])
    content_col_b = find_col(run_b, ["dripper_content", "main_content", "content"])
    is_prop_col   = find_col(run_a, ["is_propagated", "layout_template_used", "templated",
                                     "llm_called"])

    print(f"Content col A: {content_col_a}")
    print(f"Content col B: {content_col_b}")
    print(f"Propagation flag: {is_prop_col}")

    if content_col_a is None or content_col_b is None:
        print("\nContent column not found — check column names above.")
    else:
        # Merge on URL
        cols_a = ["url", content_col_a] + ([is_prop_col] if is_prop_col else [])
        if "dripper_layout_id" in run_a.columns:
            cols_a.append("dripper_layout_id")
        merged = (
            run_a[cols_a]
            .merge(
                run_b[["url", content_col_b]].rename(columns={content_col_b: "content_b"}),
                on="url", how="inner"
            )
            .rename(columns={content_col_a: "content_a"})
        )

        print(f"\nMerged A ∩ B: {len(merged):,} rows")

        # Add host info from manifest
        if manifest is not None and "url_host_name" in manifest.columns:
            host_map = manifest[["url", "url_host_name"]].drop_duplicates("url")
            if "dripper_layout_id" not in merged.columns and "dripper_layout_id" in manifest.columns:
                host_map = manifest[["url", "url_host_name", "dripper_layout_id"]].drop_duplicates("url")
            merged = merged.merge(host_map, on="url", how="left")

        # Compute F1
        merged["f1"] = [
            _token_f1(str(a or ""), str(b or ""))
            for a, b in zip(merged["content_a"], merged["content_b"])
        ]

        f1_df = merged.copy()

        print(f"\nF1 distribution (all {len(f1_df):,} rows):")
        print(f"  Mean F1:    {f1_df['f1'].mean():.4f}")
        print(f"  Median F1:  {f1_df['f1'].median():.4f}")
        print(f"  Min F1:     {f1_df['f1'].min():.4f}")
        print(f"  Max F1:     {f1_df['f1'].max():.4f}")
        print(f"  F1 >= 0.95: {(f1_df['f1'] >= 0.95).sum():,} / {len(f1_df):,}"
              f" ({(f1_df['f1'] >= 0.95).mean()*100:.1f}%)")
        print(f"  F1 >= 0.90: {(f1_df['f1'] >= 0.90).sum():,} / {len(f1_df):,}"
              f" ({(f1_df['f1'] >= 0.90).mean()*100:.1f}%)")

        if is_prop_col and is_prop_col in f1_df.columns:
            # is_propagated=True means template was used; llm_called=False means same
            if is_prop_col == "llm_called":
                prop = f1_df[f1_df[is_prop_col] == False]
                direct = f1_df[f1_df[is_prop_col] == True]
            else:
                prop = f1_df[f1_df[is_prop_col] == True]
                direct = f1_df[f1_df[is_prop_col] == False]
            print(f"\nPropagated rows ({len(prop):,}): mean F1 = {prop['f1'].mean():.4f}")
            print(f"Direct LLM rows  ({len(direct):,}): mean F1 = {direct['f1'].mean():.4f}")

In [ ]:
if f1_df is not None and len(f1_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Left: full histogram
    ax = axes[0]
    ax.hist(f1_df["f1"], bins=50, color="steelblue", edgecolor="white", linewidth=0.3)
    ax.axvline(f1_df["f1"].mean(), color="orange", linewidth=2, linestyle="--",
               label=f"Mean: {f1_df['f1'].mean():.4f}")
    ax.axvline(0.95, color="red", linewidth=1.5, linestyle=":", label="Threshold: 0.95")
    ax.set_xlabel("Token F1 (Run A vs Run B)")
    ax.set_ylabel("Pages")
    ax.set_title("F1 Distribution — All Merged Rows")
    ax.legend()
    pct_good = (f1_df["f1"] >= 0.95).mean() * 100
    ax.text(0.02, 0.97, f"{pct_good:.1f}% ≥ 0.95",
            transform=ax.transAxes, va="top", fontsize=11,
            bbox=dict(boxstyle="round", fc="#eaf4ff", ec="steelblue"))

    # Right: propagated vs direct, or CDF
    ax = axes[1]
    if is_prop_col and is_prop_col in f1_df.columns:
        if is_prop_col == "llm_called":
            prop_f1   = f1_df[f1_df[is_prop_col] == False]["f1"]
            direct_f1 = f1_df[f1_df[is_prop_col] == True]["f1"]
        else:
            prop_f1   = f1_df[f1_df[is_prop_col] == True]["f1"]
            direct_f1 = f1_df[f1_df[is_prop_col] == False]["f1"]
        ax.hist(prop_f1,   bins=40, alpha=0.7, color="#5cb85c",
                label=f"Propagated (n={len(prop_f1):,})")
        ax.hist(direct_f1, bins=40, alpha=0.7, color="#d9534f",
                label=f"Direct LLM  (n={len(direct_f1):,})")
        ax.axvline(0.95, color="black", linestyle="--", linewidth=1.2)
        ax.set_xlabel("Token F1")
        ax.set_ylabel("Pages")
        ax.set_title("F1 by Extraction Mode (propagated vs direct LLM)")
        ax.legend()
    else:
        ax.hist(f1_df["f1"], bins=60, cumulative=True, density=True, color="steelblue",
                histtype="step", linewidth=2)
        ax.axvline(0.95, color="red",    linestyle=":",  linewidth=1.5, label="F1=0.95")
        ax.axhline(0.95, color="orange", linestyle="--", linewidth=1,   label="CDF=0.95")
        ax.set_xlabel("Token F1")
        ax.set_ylabel("CDF")
        ax.set_title("F1 Cumulative Distribution")
        ax.legend()

    plt.suptitle("Quality: Run A vs Run B (standalone = ground truth)",
                 fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("F1 data not available — complete Section 1 and re-run.")

## 5. Per-Host Analysis

Which hosts saved the most LLM calls via clustering?  
Which hosts had the worst mean F1 quality?

In [ ]:
host_stats = None
host_f1    = None

if manifest is None:
    print("Manifest not loaded — skipping per-host analysis.")
else:
    # ── Calls saved per host ────────────────────────────────────────────────
    if "dripper_layout_id" in manifest.columns:
        named_m = manifest[manifest["dripper_layout_id"].str.startswith("layout-", na=False)].copy()
        cluster_sizes = named_m.groupby("dripper_layout_id").size().rename("cluster_size")
        named_m = named_m.merge(cluster_sizes, on="dripper_layout_id", how="left")
        named_m["saved_calls"] = named_m["cluster_size"] - 1  # 1 call per cluster

        host_stats = named_m.groupby("url_host_name").agg(
            total_pages  = ("url",    "count"),
            n_clusters   = ("dripper_layout_id", "nunique"),
            saved_calls  = ("saved_calls", "sum"),
        ).reset_index()
        host_stats["save_rate"] = host_stats["saved_calls"] / host_stats["total_pages"]
        host_stats = host_stats.sort_values("saved_calls", ascending=False)

        print(f"Top 15 hosts by saved LLM calls:")
        print(host_stats.head(15).to_string(index=False))
    else:
        print("dripper_layout_id not in manifest.")

    # ── F1 per host ─────────────────────────────────────────────────────────
    if f1_df is not None and "url_host_name" in f1_df.columns:
        host_f1 = (
            f1_df.groupby("url_host_name")["f1"]
            .agg(["mean", "min", "count"])
            .rename(columns={"mean": "mean_f1", "min": "min_f1", "count": "n_pages"})
            .sort_values("mean_f1")
        )
        print("\nWorst 10 hosts by mean F1:")
        print(host_f1.head(10).to_string())
        print("\nBest 10 hosts by mean F1:")
        print(host_f1.tail(10).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: top hosts by calls saved
ax = axes[0]
if host_stats is not None:
    top15 = host_stats.head(15)
    ax.barh(top15["url_host_name"], top15["saved_calls"], color="#5cb85c")
    ax.set_xlabel("LLM calls saved")
    ax.set_title("Top Hosts: LLM Calls Saved by Clustering")
    ax.invert_yaxis()
    ax.tick_params(axis="y", labelsize=8)
else:
    ax.text(0.5, 0.5, "Manifest not available",
            ha="center", va="center", transform=ax.transAxes, fontsize=11, color="gray")
    ax.set_title("Top Hosts: LLM Calls Saved")

# Right: worst hosts by F1
ax = axes[1]
if host_f1 is not None:
    worst = host_f1[host_f1["n_pages"] >= 3].head(15)
    bar_colors = ["#d9534f" if v < 0.95 else "#5cb85c" for v in worst["mean_f1"]]
    ax.barh(worst.index, worst["mean_f1"], color=bar_colors)
    ax.axvline(0.95, color="black", linestyle="--", linewidth=1.2, label="0.95")
    ax.set_xlabel("Mean F1")
    ax.set_title("Worst Hosts by Mean F1 (≥3 pages)")
    ax.invert_yaxis()
    ax.tick_params(axis="y", labelsize=8)
    ax.legend()
else:
    ax.text(0.5, 0.5, "F1 data not available",
            ha="center", va="center", transform=ax.transAxes, fontsize=11, color="gray")
    ax.set_title("Worst Hosts by Mean F1")

plt.tight_layout()
plt.show()

## 6. Cluster Size Distribution

Distribution of layout cluster sizes from the precomputed manifest.  
The mega-host (3004 pages) is highlighted — one LLM call serves 3000+ pages.

In [ ]:
vc = None
named_m = failed_m = None
max_cluster_size = 0
max_cluster_host = "N/A"

if manifest is None:
    print("Manifest not loaded — skipping cluster size analysis.")
elif "dripper_layout_id" not in manifest.columns:
    print("'dripper_layout_id' column not found in manifest.")
    print(f"Available columns: {list(manifest.columns)}")
else:
    named_m  = manifest[manifest["dripper_layout_id"].str.startswith("layout-", na=False)]
    failed_m = manifest[~manifest["dripper_layout_id"].str.startswith("layout-", na=False)]
    vc = named_m["dripper_layout_id"].value_counts()

    max_cluster_size = int(vc.max()) if len(vc) else 0
    max_cluster_id   = vc.index[0]   if len(vc) else "N/A"
    if "url_host_name" in named_m.columns and len(vc):
        max_cluster_host = named_m[
            named_m["dripper_layout_id"] == max_cluster_id
        ]["url_host_name"].iloc[0]

    print(f"Total pages:       {len(manifest):,}")
    print(f"Clustered:         {len(named_m):,} ({len(named_m)/len(manifest)*100:.1f}%)")
    print(f"Unclustered:       {len(failed_m):,} ({len(failed_m)/len(manifest)*100:.1f}%)")
    print(f"Unique clusters:   {vc.nunique():,}")
    print(f"Largest cluster:   {max_cluster_size:,} pages — {max_cluster_id}")
    print(f"Mega-host:         {max_cluster_host}")
    print()
    print("Cluster size percentiles:")
    for p in [50, 75, 90, 95, 99, 100]:
        print(f"  p{p:3d}: {vc.quantile(p/100):.0f} pages")

In [ ]:
if vc is not None and len(vc) > 0:
    max_sz  = max(int(vc.max()), 1)
    bins_edges = [1, 2, 5, 10, 25, 50, 100, 250, 500, 1000, max_sz + 1]
    bin_labels = [f"{bins_edges[i]}-{bins_edges[i+1]-1}" if bins_edges[i+1] - bins_edges[i] > 1
                  else str(bins_edges[i])
                  for i in range(len(bins_edges) - 1)]
    cluster_counts = [int(((vc >= bins_edges[i]) & (vc < bins_edges[i+1])).sum())
                      for i in range(len(bins_edges) - 1)]
    page_counts    = [int(vc[(vc >= bins_edges[i]) & (vc < bins_edges[i+1])].sum())
                      for i in range(len(bins_edges) - 1)]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Panel 1: number of clusters per size bucket
    ax = axes[0]
    bar_colors_c = ["steelblue"] * (len(cluster_counts) - 1) + ["#d9534f"]
    ax.bar(range(len(bin_labels)), cluster_counts, color=bar_colors_c,
           edgecolor="black", linewidth=0.4)
    ax.set_xticks(range(len(bin_labels)))
    ax.set_xticklabels(bin_labels, rotation=30, ha="right", fontsize=8)
    ax.set_xlabel("Cluster size (pages)")
    ax.set_ylabel("# clusters")
    ax.set_title(f"Clusters by Size ({len(vc):,} clusters total)")
    for i, v in enumerate(cluster_counts):
        if v > 0:
            ax.text(i, v + max(cluster_counts) * 0.01, str(v),
                    ha="center", va="bottom", fontsize=7)

    # Panel 2: pages per size bucket
    ax = axes[1]
    bar_colors_p = ["steelblue"] * (len(page_counts) - 1) + ["#d9534f"]
    ax.bar(range(len(bin_labels)), page_counts, color=bar_colors_p,
           edgecolor="black", linewidth=0.4, label="clustered")
    if failed_m is not None and len(failed_m) > 0:
        ax.bar([len(bin_labels)], [len(failed_m)], color="#777", label="unclustered")
        ax.set_xticks(list(range(len(bin_labels))) + [len(bin_labels)])
        ax.set_xticklabels(bin_labels + ["unclustered"], rotation=30, ha="right", fontsize=8)
    else:
        ax.set_xticks(range(len(bin_labels)))
        ax.set_xticklabels(bin_labels, rotation=30, ha="right", fontsize=8)
    ax.set_xlabel("Cluster size bucket")
    ax.set_ylabel("Total pages")
    ax.set_title("Pages by Cluster Size")
    ax.legend()
    ax.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda v, _: f"{v/1000:.0f}K" if v >= 1000 else str(int(v)))
    )

    # Annotate mega-cluster
    if max_cluster_size >= 1000:
        last_bucket_idx = len(bin_labels) - 1
        if page_counts[last_bucket_idx] > 0:
            axes[1].annotate(
                f"Mega-cluster\n{max_cluster_size:,} pages\n({max_cluster_host[:30]})",
                xy=(last_bucket_idx, page_counts[last_bucket_idx]),
                xytext=(last_bucket_idx - 2, max(page_counts) * 0.75),
                arrowprops=dict(arrowstyle="->", color="red"),
                fontsize=8, color="red"
            )

    fig.suptitle(
        f"{len(named_m):,} clustered + {len(failed_m):,} unclustered = {len(manifest):,} total"
        + (f" | largest: {max_cluster_size:,} pages ({max_cluster_host})" if max_cluster_size else ""),
        fontsize=10, y=1.02
    )
    plt.tight_layout()
    plt.show()
else:
    print("Cluster size chart not available — re-run Section 1 to load manifest.")

## 7. Example Content Comparison

For 3 pages — one from the worst-F1 tier, one from the median tier, one from the best-F1 tier —  
show Run A content, Run B content, and the F1 side by side.

In [ ]:
MAX_CHARS = 500


def show_comparison(row, tier_label, preview_chars=MAX_CHARS):
    f1   = row.get("f1", float("nan"))
    url  = str(row.get("url", "N/A"))
    host = str(row.get("url_host_name", ""))
    lid  = str(row.get("dripper_layout_id", ""))
    ca   = str(row.get("content_a") or "").strip()
    cb   = str(row.get("content_b") or "").strip()
    print("=" * 88)
    print(f"{tier_label}   F1 = {f1:.4f}")
    print(f"  URL    : {url}")
    print(f"  Host   : {host}    Layout: {lid}")
    print()
    print(f"  [Run A — clustering]")
    print(f"    {repr(ca[:preview_chars])}")
    print()
    print(f"  [Run B — standalone (ground truth)]")
    print(f"    {repr(cb[:preview_chars])}")
    print()


if f1_df is not None and len(f1_df) >= 3:
    sorted_by_f1 = f1_df.sort_values("f1").reset_index(drop=True)

    tiers = [
        ("WORST F1 (bottom)",  sorted_by_f1.head(1)),
        ("MEDIAN F1",          sorted_by_f1.iloc[[len(sorted_by_f1) // 2]]),
        ("BEST F1 (top)",      sorted_by_f1.tail(1)),
    ]

    for label, subset in tiers:
        if len(subset):
            show_comparison(subset.iloc[0], label)
else:
    print("F1 comparison requires merged results — complete Sections 1 and 4 first.")

In [ ]:
if f1_df is not None and len(f1_df) >= 3:
    sorted_by_f1 = f1_df.sort_values("f1").reset_index(drop=True)
    examples = pd.concat([
        sorted_by_f1.head(1),
        sorted_by_f1.iloc[[len(sorted_by_f1) // 2]],
        sorted_by_f1.tail(1),
    ]).reset_index(drop=True)
    example_labels = ["Worst F1", "Median F1", "Best F1"]

    fig, axes = plt.subplots(3, 2, figsize=(14, 12))
    for i, (_, row) in enumerate(examples.iterrows()):
        f1_val  = row["f1"]
        url_str = str(row["url"])[-70:]
        txt_a   = str(row.get("content_a") or "")[:MAX_CHARS]
        txt_b   = str(row.get("content_b") or "")[:MAX_CHARS]
        color   = "#5cb85c" if f1_val >= 0.95 else ("#f0ad4e" if f1_val >= 0.80 else "#d9534f")

        for j, (txt, run_lbl) in enumerate([
            (txt_a, "Run A (clustering)"),
            (txt_b, "Run B (standalone)"),
        ]):
            ax = axes[i][j]
            ax.text(0.01, 0.99, txt or "(empty)",
                    transform=ax.transAxes, va="top", ha="left",
                    fontsize=7, wrap=True, family="monospace",
                    bbox=dict(boxstyle="round", fc="#f8f8f8", ec="#cccccc"))
            ax.set_axis_off()
            ax.set_title(
                f"{example_labels[i]} — {run_lbl}   F1={f1_val:.4f}\n{url_str}",
                fontsize=8, color=color
            )

    plt.suptitle("Example Content Comparison (Run A vs Run B)", fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print("Visual comparison not available — complete Sections 1 and 4.")

## 8. Summary Scorecard

In [ ]:
def sc(v, fmt):
    """Format a scorecard value, or return 'pending'."""
    return fmt.format(v) if v else "pending"


sc_call_red  = sc(call_reduction_pct,   "{:.1f}%")
sc_tok_red   = sc(token_reduction_pct,  "{:.1f}%")
sc_tput_a    = sc(tput_a,               "{:.2f} pages/s")
sc_tput_b    = sc(tput_b,               "{:.2f} pages/s")
sc_h100_a    = sc(h100h_a,              "{:,.0f}")
sc_h100_b    = sc(h100h_b,              "{:,.0f}")
sc_cost_red  = sc(cost_reduction_pct,   "{:.1f}%")
sc_mean_f1   = f"{f1_df['f1'].mean():.4f}" if f1_df is not None else "pending"
sc_pct95     = f"{(f1_df['f1'] >= 0.95).mean()*100:.1f}%" if f1_df is not None else "pending"
sc_clust     = f"{vc.nunique():,}" if vc is not None else "pending"
sc_max_c     = f"{max_cluster_size:,} pages ({max_cluster_host})" if max_cluster_size else "pending"

scorecard = [
    ("LLM call reduction (A vs B)",    sc_call_red,  "pages that skipped GPU via template"),
    ("Token reduction (A vs B)",        sc_tok_red,   "prompt+completion tokens saved"),
    ("Throughput Run A",                sc_tput_a,    "with clustering"),
    ("Throughput Run B",                sc_tput_b,    "standalone Dripper"),
    ("Proj. H100-hours Run A",          sc_h100_a,    "full CC snapshot, 2.4B pages"),
    ("Proj. H100-hours Run B",          sc_h100_b,    "full CC snapshot, 2.4B pages"),
    ("H100-hour cost reduction",        sc_cost_red,  "vs standalone"),
    ("Mean propagation F1",             sc_mean_f1,   "Run B = ground truth"),
    ("% pages with F1 >= 0.95",         sc_pct95,     "quality threshold"),
    ("Unique layout clusters",          sc_clust,     "from manifest"),
    ("Largest cluster (mega-host)",     sc_max_c,     ""),
]

print()
print("╔" + "═"*75 + "╗")
print("║{:^75}║".format("SUMMARY SCORECARD — Layout Clustering vs Standalone Dripper"))
print("║{:^75}║".format("Run A=334943 (clustering)  |  Run B=334945 (standalone)"))
print("╠" + "═"*75 + "╣")
for metric, value, note in scorecard:
    note_s = f"  ← {note}" if note else ""
    line   = f"  {metric:<38s}  {value}"
    pad    = 75 - len(line) - len(note_s) - 1
    print(f"║{line}{' '*max(pad,1)}{note_s}║" if len(line + note_s) < 74
          else f"║  {metric:<38s}  {value:<20s}║")
print("╚" + "═"*75 + "╝")

In [ ]:
# Big-number scorecard tiles
tiles = []
if call_reduction_pct:
    tiles.append(("Call\nReduction",   f"{call_reduction_pct:.1f}%",  "#5cb85c"))
if f1_df is not None:
    tiles.append(("Mean F1",           f"{f1_df['f1'].mean():.4f}",
                  "#5cb85c" if f1_df["f1"].mean() >= 0.95 else "#f0ad4e"))
    tiles.append(("F1 ≥ 0.95",         f"{(f1_df['f1'] >= 0.95).mean()*100:.1f}%",
                  "#5cb85c" if (f1_df["f1"] >= 0.95).mean() >= 0.90 else "#f0ad4e"))
if h100h_a and h100h_b:
    tiles.append(("H100h\nRun A",  f"{h100h_a/1000:.0f}K",  "#5cb85c"))
    tiles.append(("H100h\nRun B",  f"{h100h_b/1000:.0f}K",  "#d9534f"))
if vc is not None:
    tiles.append(("Largest\nCluster", f"{max_cluster_size:,}", "#337ab7"))

if tiles:
    n   = len(tiles)
    fig, axes = plt.subplots(1, n, figsize=(3.0 * n, 3.2))
    if n == 1:
        axes = [axes]
    for ax, (label, big, color) in zip(axes, tiles):
        ax.set_facecolor(color)
        ax.text(0.5, 0.62, big,
                transform=ax.transAxes, ha="center", va="center",
                fontsize=24, fontweight="bold", color="white")
        ax.text(0.5, 0.22, label,
                transform=ax.transAxes, ha="center", va="center",
                fontsize=11, color="white", fontweight="bold")
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_edgecolor("white"); spine.set_linewidth(2)
    plt.suptitle(
        "Summary Scorecard: Layout Clustering vs Standalone Dripper"
        "  |  Run A=334943  Run B=334945",
        fontsize=11, y=1.05
    )
    plt.tight_layout()
    plt.show()
else:
    print("Scorecard tiles pending — re-run after jobs complete.")